# Permian Flaring Animation Over Time

This notebook animates how flaring (`total_vented_flared_mcf`) changes month by month across the
Permian region, for wells that are actually associated with a known VIIRS-detected flaring/combustion
site (`site_id != -1`). Each well is plotted at its real location, colored by how much gas it flared
that month, and overlaid on a light basemap of Texas.

Two GIFs come out of this notebook, both driven by the exact same underlying data and color scale, so
they're directly comparable:

1. **No blurring** — every well is drawn as its own discrete point. Well boundaries stay sharp; two
   wells sitting right next to each other never blend into one color.
2. **Merged** — each well's flared volume is spread out as a soft Gaussian glow instead of a hard dot,
   and overlapping glows from nearby wells add together, so dense clusters of flaring wells appear as
   smooth, blurred blobs of color with no visible individual dots or hard edges — the same look as a
   satellite heat/glow map.

**Source file:** `permian_prod_per_well_with_site_id.csv` — the same well-level output used by the
flaring-site-coverage notebook. `site_id != -1` marks a well whose location falls inside a known
VIIRS Nightfire combustion-site polygon (see that pipeline stage for how `site_id` is assigned).

## Why filter before loading everything into memory

`permian_prod_per_well_with_site_id.csv` has one row per well per month and can run into the tens of
millions of rows — most of which belong to wells with `site_id == -1` (no known flaring-site match)
and aren't needed here at all. Loading the whole file first and filtering afterward would mean paying
the memory cost of every row just to throw most of them away.

Instead, this notebook streams the file in fixed-size batches with `pandas.read_csv(...,
chunksize=...)`, keeps only the columns actually needed for the animation, and drops every row with
`site_id == -1` as each batch comes in. Only the much smaller `site_id != -1` subset is ever kept
around — small enough, across the whole time range, to hold comfortably in memory for building
animation frames.

## 1. Imports

In [19]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import contextily as cx
from scipy.ndimage import gaussian_filter

from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import LogNorm

pd.set_option("display.width", 120)
print("✓ Imports ready.")


✓ Imports ready.


## 2. Configuration

- `WELLS_PATH` — the well-level, site-matched CSV (same file and same relative location used by the
  flaring-site-coverage notebook).
- `COLS` — only the columns actually needed for the animation: the month (`date`), each well's
  coordinates, its flared-gas volume, and `site_id` (needed just to filter, not plotted).
- `BATCH_SIZE` — rows read per chunk while streaming and filtering the source CSV.
- `CMAP` — `"inferno"`, a perceptually-ordered colormap that runs from near-black through purple and
  orange to bright yellow at the high end — bright yellow reads as "more flaring", dark as "less
  flaring", matching the requested color scheme directly.
- `NO_BLUR_GIF` / `MERGED_GIF` — output filenames for the two animations.
- `BLUR_GRID_NX` / `BLUR_RADIUS_M` — control the *size* of each well's glow in the merged GIF: how
  fine the raster grid is, and how far (in meters) a single well's value spreads before it's added to
  its neighbors — see Section 9. This is the main knob for how "big" individual dots look once
  blurred; turn `BLUR_RADIUS_M` down for smaller, tighter glows, up for larger ones that merge more
  readily.
- `ALPHA_GAMMA` / `MAX_ALPHA` — control overall opacity of the merged glow, independent of its size —
  see Section 9.
- The Permian Basin bounding box, matching the one used throughout this pipeline, used here to frame
  the map extent.

In [20]:
WELLS_PATH = "../data/processed/texas/permian_only/permian_prod_per_well_with_site_id.csv"

COLS = ["date", "longitude", "latitude", "total_vented_flared_mcf", "site_id"]
BATCH_SIZE = 1_000_000

CMAP = "inferno"

NO_BLUR_GIF = "flaring_animation_no_blur.gif"
MERGED_GIF = "flaring_animation_merged.gif"

# Merged-GIF raster/blur settings — see Section 9
BLUR_GRID_NX = 400        # grid columns spanning the map extent (rows are scaled to keep pixels square)
BLUR_RADIUS_M = 3_500     # Gaussian blur radius in meters (~3.5 km) — how far a single well's glow spreads
ALPHA_GAMMA = 1.0         # opacity scales directly with intensity — low flaring stays visibly present, not near-invisible
MAX_ALPHA = 0.9           # opacity of the most intense glow (1 = fully solid, 0 = fully see-through)

# Permian Basin bounding box (WGS84 decimal degrees) — same region used throughout this pipeline
LAT_MIN, LAT_MAX = 29.462935, 34.021515
LONG_MIN, LONG_MAX = -105.21988, -100.036107

FPS = 4  # animation playback speed

print(f"Reading   : {WELLS_PATH}")
print(f"Output 1  : {NO_BLUR_GIF}  (discrete, no blurring)")
print(f"Output 2  : {MERGED_GIF}  (merged / blurred)")


Reading   : ../data/processed/texas/permian_only/permian_prod_per_well_with_site_id.csv
Output 1  : flaring_animation_no_blur.gif  (discrete, no blurring)
Output 2  : flaring_animation_merged.gif  (merged / blurred)


## 3. A Cheap Column Check

Reads only the CSV header (`nrows=0`) before committing to a full streaming pass, to confirm every
column this notebook needs is actually present in the file.

In [21]:
header_only = pd.read_csv(WELLS_PATH, nrows=0)
missing = [c for c in COLS if c not in header_only.columns]
if missing:
    raise ValueError(f"Expected columns not found in {WELLS_PATH}: {missing}")
print(f"✓ All {len(COLS)} needed columns are present.")


✓ All 5 needed columns are present.


## 4. Stream the CSV, Filter to `site_id != -1`, and Accumulate

Each batch is read with only `COLS` parsed out, immediately filtered down to rows where `site_id !=
-1` (a real flaring-site match), and appended to a list of small DataFrames. Rows that don't match a
site are discarded as soon as their batch is processed — they're never accumulated. Once every batch
has been read, the kept batches are concatenated into a single DataFrame that holds only the rows
this notebook actually needs.

In [22]:
kept_batches = []
total_rows_seen = 0

reader = pd.read_csv(WELLS_PATH, usecols=COLS, parse_dates=["date"], chunksize=BATCH_SIZE)

for i, chunk in enumerate(reader, 1):
    total_rows_seen += len(chunk)
    matched = chunk.loc[chunk["site_id"] != -1].copy()
    if len(matched):
        kept_batches.append(matched)
    print(f"  batch {i:3d}: {len(chunk):,} rows read, {len(matched):,} site-matched rows kept "
          f"(running total kept: {sum(len(b) for b in kept_batches):,})")

wells = pd.concat(kept_batches, ignore_index=True)
del kept_batches

print(f"\nTotal rows read     : {total_rows_seen:,}")
print(f"Site-matched rows   : {len(wells):,} ({len(wells) / total_rows_seen:.2%} of all rows)")
print(f"Unique locations    : {wells[['longitude', 'latitude']].drop_duplicates().shape[0]:,}")
wells.head()


  batch   1: 1,000,000 rows read, 105,364 site-matched rows kept (running total kept: 105,364)
  batch   2: 1,000,000 rows read, 143,528 site-matched rows kept (running total kept: 248,892)
  batch   3: 1,000,000 rows read, 97,964 site-matched rows kept (running total kept: 346,856)
  batch   4: 1,000,000 rows read, 161,065 site-matched rows kept (running total kept: 507,921)
  batch   5: 1,000,000 rows read, 100,514 site-matched rows kept (running total kept: 608,435)
  batch   6: 1,000,000 rows read, 148,853 site-matched rows kept (running total kept: 757,288)
  batch   7: 1,000,000 rows read, 108,149 site-matched rows kept (running total kept: 865,437)
  batch   8: 1,000,000 rows read, 133,130 site-matched rows kept (running total kept: 998,567)
  batch   9: 1,000,000 rows read, 130,485 site-matched rows kept (running total kept: 1,129,052)
  batch  10: 1,000,000 rows read, 132,472 site-matched rows kept (running total kept: 1,261,524)
  batch  11: 1,000,000 rows read, 150,778 site-

,total_vented_flared_mcf,date,longitude,latitude,site_id
0,0.0,2012-02-01,-102.991182,31.563510,9511
1,0.0,2012-02-01,-102.976979,31.543236,9523
2,0.0,2012-02-01,-102.976979,31.543236,9523
3,0.0,2012-02-01,-102.964586,31.557178,9523
4,0.0,2012-02-01,-102.970768,31.555464,9523


## 5. Build Monthly Frames and Project Coordinates

Each row's `date` is collapsed to its calendar month (`period`), giving one animation frame per
month. Well coordinates come in as plain WGS84 longitude/latitude, but the basemap tiles used later
(Section 7) are served in Web Mercator (EPSG:3857) — so points are converted to a GeoDataFrame and
reprojected to EPSG:3857 once, up front, rather than repeating that conversion inside every animation
frame.

In [23]:
wells["period"] = wells["date"].dt.to_period("M").dt.to_timestamp()

wells_gdf = gpd.GeoDataFrame(
    wells.drop(columns=["longitude", "latitude"]),
    geometry=gpd.points_from_xy(wells["longitude"], wells["latitude"]),
    crs="EPSG:4326",
)
wells_3857 = wells_gdf.to_crs(epsg=3857)

months = sorted(wells_3857["period"].unique())
print(f"Frames (months): {len(months)}  ({pd.Timestamp(months[0]):%Y-%m} → {pd.Timestamp(months[-1]):%Y-%m})")


Frames (months): 170  (2012-02 → 2026-03)


## 6. Shared Color Scale

Flared-gas volumes are heavily right-skewed — a small number of wells flare far more than the rest —
so a **log** color scale (`LogNorm`) is used rather than a linear one; on a linear scale, all but the
very largest flares would look identically dark.

The same `vmin`/`vmax` are used for every frame of both animations (computed once, here, across the
entire site-matched dataset) rather than being recomputed per frame. If each frame rescaled its own
color range, a quiet month and a heavy-flaring month could end up looking equally "bright yellow" —
defeating the point of a color scale that's supposed to be comparable over time. `vmax` is capped at
the 99th percentile (rather than the raw max) so that one extreme outlier month doesn't wash out the
color contrast for every other month; `vmin` is floored at 1 to keep `LogNorm` well-defined.

In [24]:
positive_flare = wells_3857.loc[wells_3857["total_vented_flared_mcf"] > 0, "total_vented_flared_mcf"]

VMAX = positive_flare.quantile(0.99)
VMIN = max(1.0, positive_flare.quantile(0.05))

norm = LogNorm(vmin=VMIN, vmax=VMAX)

print(f"Color scale (log): vmin={VMIN:,.1f} MCF → vmax={VMAX:,.1f} MCF")
print("Bright yellow = near or above vmax MCF flared that month. Dark = near vmin or no flaring.")


Color scale (log): vmin=1.0 MCF → vmax=3,891.2 MCF
Bright yellow = near or above vmax MCF flared that month. Dark = near vmin or no flaring.


## 7. Basemap Extent

Both animations share the same map extent — the site-matched wells' own bounding box, padded by 5%
on each side — and the same light, near-white Texas basemap
(`contextily.providers.CartoDB.Positron`), so the two GIFs are visually aligned and directly
comparable frame for frame.

Fetching basemap tiles requires an internet connection at run time. If tiles can't be fetched (e.g.
no network access), the notebook falls back to a plain white background instead of failing outright
— the animation itself still renders, just without the underlying map imagery.

In [28]:
xmin, ymin, xmax, ymax = wells_3857.total_bounds
xpad = (xmax - xmin) * 0.05
ypad = (ymax - ymin) * 0.05
XLIM = (xmin - xpad, xmax + xpad)
YLIM = (ymin - ypad, ymax + ypad)

BASEMAP_SOURCE = cx.providers.CartoDB.DarkMatterNoLabels  # light, near-white basemap


def add_basemap_safely(ax):
    """Add the light Texas basemap; fall back to a plain white background if tiles can't be fetched."""
    try:
        cx.add_basemap(ax, source=BASEMAP_SOURCE, attribution_size=6)
        return True
    except Exception as exc:
        print(f"  ⚠ Could not fetch basemap tiles ({exc!r}); falling back to a plain white background.")
        ax.set_facecolor("white")
        return False


print(f"Map extent (EPSG:3857): x=[{XLIM[0]:,.0f}, {XLIM[1]:,.0f}]  y=[{YLIM[0]:,.0f}, {YLIM[1]:,.0f}]")


Map extent (EPSG:3857): x=[-11,689,104, -11,152,866]  y=[3,542,940, 4,030,406]


## 8. GIF 1 — No Blurring (Discrete Per-Well Markers)

Every site-matched well is drawn as its own point, sized constantly and colored only by that month's
`total_vented_flared_mcf` on the shared log scale from Section 6. There's no spatial smoothing or
interpolation between nearby wells — two wells sitting right next to each other each keep their own
distinct color, so well boundaries never blend together.

In [37]:
fig1, ax1 = plt.subplots(figsize=(10, 9))
ax1.set_xlim(*XLIM)
ax1.set_ylim(*YLIM)
add_basemap_safely(ax1)
ax1.set_axis_off()
fig1.patch.set_facecolor("black")

scatter = ax1.scatter(
    [], [],
    s=28,
    c=[],
    cmap=CMAP,
    norm=norm,
    alpha=0.9,
    edgecolors="black",
    linewidths=0.3,
)

title1 = ax1.set_title("", fontsize=15, pad=12, color = 'white')

cbar1 = fig1.colorbar(scatter, ax=ax1, shrink=0.7)
cbar1.set_label("Gas flared this month (MCF, log scale)",color = 'white')
cbar1.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar1.ax.get_yticklabels(), color="white")


def update_no_blur(frame_index):
    month = months[frame_index]
    frame = wells_3857.loc[wells_3857["period"] == month]

    x = frame.geometry.x.values
    y = frame.geometry.y.values
    flared = frame["total_vented_flared_mcf"].clip(lower=VMIN, upper=VMAX).values

    scatter.set_offsets(np.column_stack([x, y]) if len(frame) else np.empty((0, 2)))
    scatter.set_array(flared)

    title1.set_text(
        f"Permian Flaring by Well — {pd.Timestamp(month):%B %Y}  "
        f"({len(frame):,} site-matched wells)"
    )
    return scatter, title1


anim_no_blur = FuncAnimation(fig1, update_no_blur, frames=len(months), interval=1000 / FPS, blit=False)
anim_no_blur.save(NO_BLUR_GIF, writer=PillowWriter(fps=FPS))
plt.close(fig1)

print(f"✓ Saved {NO_BLUR_GIF}")


✓ Saved flaring_animation_no_blur.gif


## 9. GIF 2 — Merged (Wells Blurred Into Glowing, Overlapping Blobs)

Same data, same color scale, same map extent as GIF 1 — but instead of plotting each well as its own
hard-edged dot, each well's flared volume is first rasterized onto a fine grid, then passed through a
**Gaussian blur** (`scipy.ndimage.gaussian_filter`). This spreads every well's value out into a soft,
fading glow rather than a sharp point. Where two wells sit close together, their glows physically
overlap and add together in the same grid cells, so dense flaring clusters merge into one smooth,
continuous blob of color with no visible dot edges or bin boundaries — closer to a satellite heat/glow
map than to individually plotted points.

Concretely, for each month:

1. **Rasterize** — `np.histogram2d(..., weights=flared_values)` sums each well's flared volume into
   the grid cell it falls in, producing a grid that's mostly zero with a few "spikes" at well
   locations.
2. **Blur** — `gaussian_filter(grid, sigma=...)` smooths those spikes into soft, overlapping glows.
   `sigma` is derived from `BLUR_RADIUS_M` (Section 2) converted into grid-cell units, so the blur
   radius stays a fixed physical distance regardless of grid resolution. A Gaussian blur kernel is
   normalized to sum to 1 (so it doesn't add energy out of nowhere), which means it also *dilutes* a
   sharp spike's peak brightness by spreading it over a wider area — left alone, every well would
   fade into a dim smudge instead of a bright glow. To counteract that dilution and keep a single
   well's peak brightness close to its actual flared value, the blurred grid is rescaled by
   `2 * pi * sigma^2` (the standard normalization factor for a 2D Gaussian). Wells that are close
   enough for their glows to overlap still add together and shine brighter than either alone — that
   additive brightening *is* the "merging" effect — while an isolated well's glow stays close to its
   own true intensity rather than washing out.
3. **Color + fade to transparent** — the rescaled, blurred grid is mapped through the same log color
   scale as GIF 1 (`inferno`, bright yellow = more flaring), but grid cells with little or no flaring
   are also made increasingly *transparent* rather than solid dark, so the light Texas basemap still
   shows through everywhere nothing is flaring — only areas with real flaring glow with color.

   Opacity is tied to the same 0–1 normalized intensity used for color (`normed`, from the shared log
   scale in Section 6), raised to `ALPHA_GAMMA` and capped at `MAX_ALPHA` (both set in Section 2); by
   default (`ALPHA_GAMMA = 1.0`) opacity scales directly with intensity, so even low-flaring wells
   stay visibly present rather than fading to near-nothing.

   **If the merged GIF feels like it's overwhelming the basemap, the main knob to reach for is
   `BLUR_RADIUS_M` in Section 2** — it directly controls how large a footprint each individual well's
   glow covers before it's added to its neighbors. A smaller radius means smaller, tighter glows with
   more white space between wells that aren't actually close together; a larger radius means bigger
   glows that merge together more readily (and more of the map ends up covered by some color).

In [39]:
fig2, ax2 = plt.subplots(figsize=(10, 9))
ax2.set_xlim(*XLIM)
ax2.set_ylim(*YLIM)
add_basemap_safely(ax2)
ax2.set_axis_off()
fig2.patch.set_facecolor("black")

title2 = ax2.set_title("", fontsize=15, pad=12, color = 'white')

# Build the raster grid once — same grid is reused (and overwritten) every frame.
aspect = (XLIM[1] - XLIM[0]) / (YLIM[1] - YLIM[0])
grid_nx = BLUR_GRID_NX
grid_ny = max(1, int(round(grid_nx / aspect)))

x_edges = np.linspace(XLIM[0], XLIM[1], grid_nx + 1)
y_edges = np.linspace(YLIM[0], YLIM[1], grid_ny + 1)

cell_size_m = (XLIM[1] - XLIM[0]) / grid_nx
blur_sigma_cells = BLUR_RADIUS_M / cell_size_m

cmap_obj = plt.get_cmap(CMAP)

# A colorbar can be attached to a plain ScalarMappable — nothing needs to be drawn first.
sm = cm.ScalarMappable(cmap=CMAP, norm=norm)
cbar2 = fig2.colorbar(sm, ax=ax2, shrink=0.7)
cbar2.set_label("Gas flared this month (MCF, log scale, blurred)",color = 'white')
cbar2.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar2.ax.get_yticklabels(), color="white")

img_artist = ax2.imshow(
    np.zeros((grid_ny, grid_nx, 4)),
    extent=(XLIM[0], XLIM[1], YLIM[0], YLIM[1]),
    origin="lower",
    interpolation="bilinear",
)


def update_merged(frame_index):
    month = months[frame_index]
    frame = wells_3857.loc[wells_3857["period"] == month]

    title2.set_text(
        f"Permian Flaring, Merged — {pd.Timestamp(month):%B %Y}  "
        f"({len(frame):,} site-matched wells)"
    )

    if len(frame) == 0:
        img_artist.set_data(np.zeros((grid_ny, grid_nx, 4)))
        return img_artist, title2

    x = frame.geometry.x.values
    y = frame.geometry.y.values
    weights = frame["total_vented_flared_mcf"].clip(lower=0).values

    # Step 1: rasterize — sum each well's flared volume into its grid cell.
    grid, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges], weights=weights)

    # Step 2: blur — spread each cell's value into a soft glow that overlaps with its neighbors, then
    # rescale to undo the peak-brightness dilution a normalized Gaussian kernel otherwise causes.
    blurred = gaussian_filter(grid, sigma=blur_sigma_cells) * (2 * np.pi * blur_sigma_cells ** 2)

    # Step 3: color + fade to transparent for near-zero cells.
    normed = np.clip(norm(np.clip(blurred, VMIN, None)), 0, 1)
    rgba = cmap_obj(normed)
    # ALPHA_GAMMA > 1 pushes faint/background glow toward fully transparent faster than the color
    # itself fades, so a wide halo of low-intensity blur doesn't visually blanket the whole map.
    rgba[..., 3] = (normed ** ALPHA_GAMMA) * MAX_ALPHA

    img_artist.set_data(np.transpose(rgba, (1, 0, 2)))
    return img_artist, title2


anim_merged = FuncAnimation(fig2, update_merged, frames=len(months), interval=1000 / FPS, blit=False)
anim_merged.save(MERGED_GIF, writer=PillowWriter(fps=FPS))
plt.close(fig2)

print(f"✓ Saved {MERGED_GIF}")


✓ Saved flaring_animation_merged.gif


## 10. Summary

This notebook produced two GIFs, both animating `total_vented_flared_mcf` month by month
(2012–2026) across the Permian region, restricted to wells with a real `site_id`:

| File | Style |
|---|---|
| `flaring_animation_no_blur.gif` | Each well is its own discrete point — no blending between nearby wells. |
| `flaring_animation_merged.gif` | Each well's value is rasterized and Gaussian-blurred into a soft glow; overlapping glows from nearby wells add together into smooth, merged blobs of color. |

Both share the same **log** color scale (`inferno`, bright yellow = more flaring, dark = less), the
same fixed `vmin`/`vmax` across every frame so brightness is comparable month to month, and the same
map extent over a light, near-white Texas basemap — so the two are a direct, frame-by-frame
comparison of "individual wells" vs. "merged regional glow" views of the same underlying data.